# V21 pre-freeze gates E1/E2/E5 (engineering, VALIDATION data only): official Kwon/K-prune with local tokenizer dirs (same-kernel A/B on BERT-base MRPC), V20 MuRIL teacher, mBERT random head, add-on extra-set dumps. docs/V21_PREREG_DRAFT_2026-11-18.md

In [ ]:
%%bash
set -e
exec > >(tee -a /kaggle/working/run.log) 2>&1
cd /kaggle/working && rm -rf quantum
git clone -q -b research/qfc-v21-multilingual-confirm https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git log --oneline -1 | tee /kaggle/working/commit.txt
O=/kaggle/working/v21; mkdir -p $O
META=$(find /kaggle/input -name ft_meta.json 2>/dev/null | head -1); IN=$(dirname $(dirname $META)); echo "IN=$IN" | tee $O/input_root.txt
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | tee $O/gpu.txt
# the V20 MuRIL teacher (seed 42): sha256 must equal the value recorded in PROJECT_STATE
SHA=e19f3b747f4a4cadd96c4d4849b74e42222325eadc393b0149dd6d590350c901
echo "$SHA  $IN/ft/mnli/pytorch_model.bin" | sha256sum -c - || { echo TEACHER_SHA_MISMATCH; exit 1; }
cp -r $IN/ft/mnli /tmp/teacher
export V21_TEACHER_DIR=/tmp/teacher V21_V20_IN=$IN
bash scripts/v15_kaggle.sh v21e1 /tmp/v15 $O > $O/v21e1_stage.log 2>&1 || echo STAGE_FAILED
grep -v -i "warn\|xet" $O/v21e1_stage.log | grep -i "fail\|error\|E1_pass\|ok\|NONE\|Traceback" | cut -c1-240 | tail -60
cp /kaggle/working/run.log $O/run.log; ls -la $O $O/v21e1 || true
